# Tutorial 1: Identifiers & Preprocessing

This notebook introduces the **identifier handling** and **preprocessing** tools
in `embpy`. Before computing any embeddings you typically need to:

1. Classify what kind of identifier you have (gene symbol, Ensembl ID, SMILES, …).
2. Resolve identifiers to canonical forms (e.g. gene symbol → Ensembl ID).
3. Fetch sequences or descriptions needed by the embedding models.

`embpy` provides three main classes for this:

| Class | Purpose |
|---|---|
| `GeneResolver` | Map gene identifiers ↔ DNA / protein sequences / descriptions |
| `DrugResolver` | Map drug names ↔ SMILES strings via PubChem |
| `PerturbationProcessor` | High-level preprocessing for mixed perturbation lists |

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)  # keep output clean

## 1. Automatic identifier classification

`detect_identifier_type` inspects a string and returns one of:
`"symbol"`, `"ensembl_id"`, `"dna_sequence"`, `"protein_sequence"`, or `"smiles"`.

In [ ]:
from embpy.resources.gene_resolver import detect_identifier_type

examples = [
    "TP53",                          # gene symbol
    "ENSG00000141510",               # Ensembl gene ID
    "ACGTACGTACGTACGTACGTACGT",      # raw DNA
    "MTEYKLVVVGAGGVGKSALT",          # amino-acid sequence
    "CC(=O)O",                        # acetic acid SMILES
    "c1ccccc1",                       # benzene SMILES
]

for ex in examples:
    print(f"{ex:40s} → {detect_identifier_type(ex)}")

## 2. GeneResolver – mapping gene identifiers

`GeneResolver` uses pyensembl (local) and REST APIs (Ensembl, MyGene.info,
UniProt) to convert between gene symbols, Ensembl IDs, DNA sequences,
protein sequences, and textual descriptions.

In [ ]:
from embpy.resources.gene_resolver import GeneResolver

resolver = GeneResolver(auto_download=False)  # skip pyensembl download for speed

### 2.1 Symbol ↔ Ensembl ID

In [ ]:
# Symbol → Ensembl ID
ens_id = resolver.symbol_to_ensembl("TP53")
print(f"TP53 → {ens_id}")

# Ensembl ID → Symbol
symbol = resolver.ensembl_to_symbol("ENSG00000141510")
print(f"ENSG00000141510 → {symbol}")

In [ ]:
# Batch mapping
genes = ["TP53", "BRCA1", "EGFR", "MYC"]
mapping = resolver.symbols_to_ensembl_batch(genes)

for sym, ens in mapping.items():
    print(f"  {sym:10s} → {ens}")

### 2.2 Fetching DNA sequences

In [ ]:
dna_seq = resolver.get_dna_sequence("TP53", id_type="symbol")
if dna_seq:
    print(f"TP53 DNA length: {len(dna_seq):,} bp")
    print(f"First 100 bp:   {dna_seq[:100]}…")

### 2.3 Fetching protein sequences

In [ ]:
prot_seq = resolver.get_protein_sequence("TP53", id_type="symbol")
if prot_seq:
    print(f"TP53 protein length: {len(prot_seq)} aa")
    print(f"First 60 aa:         {prot_seq[:60]}…")

### 2.4 Gene descriptions (for text embeddings)

In [ ]:
desc = resolver.get_gene_description(
    "TP53",
    id_type="symbol",
    format_string="Gene: {symbol}. Name: {name}. Summary: {summary}",
)
if desc:
    print(desc[:300], "…")

## 3. DrugResolver – mapping drug names and SMILES

`DrugResolver` queries PubChem to convert between common drug names and
canonical SMILES strings.

In [ ]:
from embpy.resources.drug_resolver import DrugResolver

drug_resolver = DrugResolver()

In [ ]:
# Drug name → SMILES
drugs = ["aspirin", "ibuprofen", "caffeine"]
for name in drugs:
    smi = drug_resolver.name_to_smiles(name)
    print(f"  {name:12s} → {smi}")

In [ ]:
# SMILES → drug names
aspirin_smiles = "CC(=O)Oc1ccccc1C(O)=O"
names = drug_resolver.smiles_to_names(aspirin_smiles, top_k=5)
print(f"Names for aspirin SMILES: {names}")

## 4. PerturbationProcessor – high-level preprocessing

When you have a mixed list of perturbations (genes + drugs), the
`PerturbationProcessor` can classify and resolve them all at once.

In [ ]:
from embpy.pp.basic import PerturbationProcessor

pp = PerturbationProcessor()

### 4.1 resolve_identifiers – classify & canonicalize a mixed list

In [ ]:
mixed_ids = [
    "TP53",                          # gene symbol
    "ENSG00000141510",               # Ensembl ID
    "CC(=O)Oc1ccccc1C(O)=O",         # aspirin SMILES
    "ACGTACGTACGTACGTACGTACGT",      # raw DNA sequence
    "MTEYKLVVVGAGGVGKSALT",          # protein sequence
]

df = pp.resolve_identifiers(mixed_ids)
df

### 4.2 normalize_gene_names – standardize gene identifiers

In [ ]:
gene_ids = ["TP53", "ENSG00000141510.5", "BRCA1", "EGFR"]
normalized = pp.normalize_gene_names(gene_ids)

for original, canonical in normalized.items():
    print(f"  {original:25s} → {canonical}")

### 4.3 Loading genes from an AnnData file

If you have a single-cell dataset, `GeneResolver` can extract gene
identifiers directly from the `.h5ad` file.

In [ ]:
import tempfile, os
import numpy as np
import pandas as pd
import anndata as ad

# Create a small example AnnData
var = pd.DataFrame(
    {"ensembl_id": ["ENSG00000141510", "ENSG00000012048", "ENSG00000146648"]},
    index=pd.Index(["TP53", "BRCA1", "EGFR"]),
)
adata = ad.AnnData(np.random.randn(5, 3).astype(np.float32), var=var)

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "example.h5ad")
    adata.write_h5ad(path)

    # Auto-detect the ensembl_id column
    genes = resolver.load_genes_from_adata(path)
    print(f"Loaded {len(genes)} genes: {genes}")

    # Or use the index (gene symbols)
    genes_idx = resolver.load_genes_from_adata(path, column=None)
    # (falls back to var index if no known column found;
    #  here ensembl_id is found first)

## 5. Loading benchmark datasets

`embpy` ships with two complementary backends for fetching ready-to-use
perturbation datasets:

| Backend | Access pattern | Backed by |
|---|---|---|
| `embpy.dt` | `embpy.dt.replogle()` | Hugging Face Hub |
| `embpy.pp.load_lamin()` | `embpy.pp.load_lamin("mcfarland")` | LaminDB |

Both return an `AnnData` with metadata attached in `.uns`, so you can
plug the result directly into `embpy.tl.run_pipeline`.

### 5.1 Hugging Face datasets (`embpy.dt`)

Every registered dataset has its own convenience function.  Call
`list_datasets()` to see what is available, and `info()` to inspect
metadata before downloading.

In [ ]:
import embpy

# See all registered HF datasets
print("Available HF datasets:", embpy.dt.list_datasets())

In [ ]:
# Inspect a dataset card before downloading
card = embpy.dt.info("replogle")
print(f"Name:          {card.name}")
print(f"Description:   {card.description}")
print(f"Pert. column:  {card.perturbation_column}")
print(f"Use case:      {card.use_case}")
print(f"Organism:      {card.organism}")
print(f"Reference:     {card.reference}")

In [ ]:
# Download via the per-dataset shorthand
# adata = embpy.dt.replogle()

# Or via the generic loader (equivalent)
# adata = embpy.dt.load("replogle")

# The returned AnnData carries the dataset card in .uns:
# adata.uns["dataset_card"]
# {'name': 'replogle', 'perturbation_column': 'gene', 'use_case': 'protein', ...}

### 5.2 LaminDB datasets (`embpy.pp.load_lamin`)

For datasets stored in a LaminDB instance, `load_lamin` wraps
`ln.Artifact.get(uid).load()` behind a friendly name.  LaminDB must be
installed and connected to the relevant instance.

In [ ]:
# See all registered LaminDB datasets
print("Available LaminDB datasets:", embpy.pp.list_lamin_datasets())

In [ ]:
# Inspect the LaminDB dataset card
lamin_card = embpy.pp.lamin_info("mcfarland")
print(f"Name:          {lamin_card.name}")
print(f"Artifact UID:  {lamin_card.uid}")
print(f"Description:   {lamin_card.description}")
print(f"Pert. column:  {lamin_card.perturbation_column}")
print(f"Use case:      {lamin_card.use_case}")
print(f"Reference:     {lamin_card.reference}")

In [ ]:
# Load the dataset (requires lamindb installed and connected)
# Under the hood this calls: ln.Artifact.get("rCNr6NMFJRo0cyzl0000").load()
# adata = embpy.pp.load_lamin("mcfarland")

# Use cache=True for repeated local access without re-downloading
# adata = embpy.pp.load_lamin("mcfarland", cache=True)

# The returned AnnData carries the lamin card in .uns:
# adata.uns["lamin_card"]
# {'name': 'mcfarland', 'uid': 'rCNr6NMFJRo0cyzl0000', 'perturbation_column': 'gene', ...}

### 5.3 From dataset to pipeline

Both loaders attach metadata in `.uns`, so you can go straight from
loading to benchmarking without manually specifying column names:

In [ ]:
# Example: load a dataset and feed it directly into the pipeline
#
# adata = embpy.dt.replogle()
# card  = adata.uns["dataset_card"]
#
# results = embpy.tl.run_pipeline(
#     adata,
#     perturbation_column=card["perturbation_column"],
#     use_case=card["use_case"],
#     embedding_models=["esm2_650M", "esmc_300m"],
# )
#
# The same pattern works for LaminDB datasets:
#
# adata = embpy.pp.load_lamin("mcfarland")
# card  = adata.uns["lamin_card"]
#
# results = embpy.tl.run_pipeline(
#     adata,
#     perturbation_column=card["perturbation_column"],
#     use_case=card["use_case"],
#     embedding_models="all",
# )

## Summary

| Function / Class | What it does |
|---|---|
| `detect_identifier_type(s)` | Classify a string as symbol / ensembl_id / dna / protein / smiles |
| `GeneResolver.symbol_to_ensembl()` | Gene symbol → Ensembl ID |
| `GeneResolver.get_dna_sequence()` | Fetch genomic DNA for a gene |
| `GeneResolver.get_protein_sequence()` | Fetch canonical protein sequence |
| `GeneResolver.get_gene_description()` | Fetch text description from MyGene.info |
| `GeneResolver.load_genes_from_adata()` | Extract gene list from `.h5ad` file |
| `DrugResolver.name_to_smiles()` | Drug name → SMILES |
| `DrugResolver.smiles_to_names()` | SMILES → drug name(s) |
| `PerturbationProcessor.resolve_identifiers()` | Classify & resolve a mixed list |
| `PerturbationProcessor.normalize_gene_names()` | Standardize gene IDs to Ensembl |
| `embpy.dt.list_datasets()` | List registered Hugging Face datasets |
| `embpy.dt.info(name)` | Inspect a HF dataset card before downloading |
| `embpy.dt.load(name)` / `embpy.dt.<name>()` | Download an HF dataset as AnnData |
| `embpy.pp.list_lamin_datasets()` | List registered LaminDB datasets |
| `embpy.pp.lamin_info(name)` | Inspect a LaminDB dataset card |
| `embpy.pp.load_lamin(name)` | Load a dataset from LaminDB by friendly name |

Next: [Tutorial 2 – Gene (DNA) Embeddings](02_gene_embeddings.ipynb)